# Commodity Price Signal Backtester

This notebook builds a quantitative backtesting framework for commodity futures from scratch. We will analyze three major commodities:
- **Crude Oil (`CL=F`)**
- **Natural Gas (`NG=F`)**
- **Corn (`ZC=F`)**

We will implement two strategies: Momentum (Moving Average Crossover) and Mean-Reversion (Z-Score).

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 7)

## 1. Data Acquisition

We use `yfinance` to fetch historical daily price data for our commodity futures. The function `fetch_commodity_data` downloads the data and calculates daily returns.

In [ ]:
def fetch_commodity_data(tickers, start_date, end_date):
    """
    Fetches historical daily price data for a list of tickers.
    
    Args:
        tickers (list or str): Ticker symbol(s) to fetch.
        start_date (str): Start date in 'YYYY-MM-DD' format.
        end_date (str): End date in 'YYYY-MM-DD' format.
        
    Returns:
        pd.DataFrame: A DataFrame containing the 'Adj Close' prices for the tickers.
    """
    print(f"Fetching data for {tickers} from {start_date} to {end_date}...")
    df = yf.download(tickers, start=start_date, end=end_date)['Adj Close']
    
    # If a single ticker is passed, yfinance returns a Series instead of a DataFrame.
    # We convert it to a DataFrame to maintain consistency.
    if isinstance(df, pd.Series):
        df = df.to_frame(name=tickers if isinstance(tickers, str) else tickers[0])
        
    # Drop any rows with entirely missing data (e.g. holidays)
    df.dropna(how='all', inplace=True)
    
    # Forward fill any missing intermediate data points
    df.ffill(inplace=True)
    
    print("Data fetching complete.")
    return df

In [ ]:
# Let's test the data fetching for our 3 chosen commodities from 2020 to present
tickers = ['CL=F', 'NG=F', 'ZC=F']
df_prices = fetch_commodity_data(tickers, '2020-01-01', '2026-06-25')
df_prices.head()

## 2. Momentum Strategy

The momentum strategy aims to ride trends. We will use a classic **Moving Average Crossover** system.
- **Fast Moving Average (e.g. 20-day)**: Responds quickly to price changes.
- **Slow Moving Average (e.g. 50-day)**: Represents the longer-term trend.

**Signal Logic**:
- **Buy (1)**: When the Fast MA is above the Slow MA.
- **Sell/Short (-1)**: When the Fast MA is below the Slow MA.

In [ ]:
def calculate_momentum_signal(prices, fast_window=20, slow_window=50):
    """
    Calculates momentum trading signals based on Moving Average Crossover.
    
    Args:
        prices (pd.Series or pd.DataFrame): Historical prices.
        fast_window (int): Lookback period for the fast moving average.
        slow_window (int): Lookback period for the slow moving average.
        
    Returns:
        pd.DataFrame: A DataFrame containing the trading signals (1 for long, -1 for short).
    """
    # Calculate simple moving averages
    fast_ma = prices.rolling(window=fast_window).mean()
    slow_ma = prices.rolling(window=slow_window).mean()
    
    # Generate signals: 1 if fast_ma > slow_ma, else -1
    signals = np.where(fast_ma > slow_ma, 1.0, -1.0)
    
    # Convert numpy array back to pandas DataFrame/Series
    if isinstance(prices, pd.DataFrame):
        signals = pd.DataFrame(signals, index=prices.index, columns=prices.columns)
    else:
        signals = pd.Series(signals, index=prices.index, name='Signal')
        
    # Set the initial warm-up period to 0 (no position)
    signals.iloc[:slow_window] = 0.0
    
    return signals

In [ ]:
# Test the momentum signal generator
momentum_signals = calculate_momentum_signal(df_prices, fast_window=20, slow_window=50)
momentum_signals.tail()